# GWAS-Enformer-EpiAgent Workflow Demo

This notebook demonstrates the complete integrative analysis workflow for functionally validating disease-associated SNPs.

## Workflow Overview

1. **Phase 1: Hypothesis Generation**
   - GWAS data analysis to identify candidate SNPs
   - Enformer predictions for functional impact
   - Candidate enhancer identification

2. **Phase 2: Validation & Contextualization**
   - Single-cell ATAC-seq data preprocessing
   - EpiAgent-based cell type identification
   - Cell-type specific enhancer validation

3. **Phase 3: Integration & Reporting**
   - Comprehensive analysis report generation

## Setup

In [ ]:
import sys
import os
from pathlib import Path
import configparser
import subprocess
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add workflow directory to path
workflow_dir = Path('../')
sys.path.append(str(workflow_dir))

# Configuration
config_file = workflow_dir / 'config' / 'config.ini'
print(f"Using configuration: {config_file}")

## Load Configuration

In [ ]:
# Load configuration
config = configparser.ConfigParser()
config.read(config_file)

# Display key parameters
print("Analysis Configuration:")
print(f"Disease: {config['disease']['name']}")
print(f"Target tissue: {config['disease']['target_tissue']}")
print(f"Dataset ID: {config['data']['scatac_dataset_id']}")
print(f"Output directory: {config['data']['output_dir']}")

## Phase 1: GWAS & Enformer Analysis

In [ ]:
# Run Phase 1 scripts
phase1_dir = workflow_dir / 'phase1_gwas_enformer'
disease_name = config['disease']['name']

print("Running Phase 1: GWAS & Enformer Analysis")
print("="*50)

# Step 1.1: GWAS Analysis
print("Step 1.1: GWAS Analysis...")
cmd = [sys.executable, str(phase1_dir / 'gwas_analysis.py'), 
       '--config', str(config_file), '--disease', disease_name]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ GWAS analysis completed")
else:
    print(f"✗ GWAS analysis failed: {result.stderr}")

In [ ]:
# Load GWAS results
output_dir = Path(config['data']['output_dir'])
gwas_summary_file = output_dir / f"{disease_name.replace(' ', '_')}_analysis_summary.json"

if gwas_summary_file.exists():
    with open(gwas_summary_file, 'r') as f:
        gwas_summary = json.load(f)
    
    print("GWAS Analysis Results:")
    print(f"Lead SNP: {gwas_summary['lead_snp']}")
    print(f"Position: chr{gwas_summary['lead_snp_chr']}:{gwas_summary['lead_snp_pos']:,}")
    print(f"P-value: {gwas_summary['lead_snp_pvalue']:.2e}")
    print(f"Candidate SNPs: {gwas_summary['n_candidate_snps']}")
    
    target_snp = gwas_summary['lead_snp']
else:
    print("GWAS summary not found")
    target_snp = "rs1000000"  # Default for demo

In [ ]:
# Continue with sequence preparation and Enformer prediction
print("\nStep 1.2: Sequence Preparation...")
cmd = [sys.executable, str(phase1_dir / 'prepare_enformer_sequences.py'),
       '--config', str(config_file), '--target-snp', target_snp]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Sequence preparation completed")
else:
    print(f"✗ Sequence preparation failed: {result.stderr}")

In [ ]:
print("\nStep 1.3: Enformer Predictions...")
cmd = [sys.executable, str(phase1_dir / 'run_enformer_predictions.py'),
       '--config', str(config_file), '--target-snp', target_snp]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Enformer predictions completed")
else:
    print(f"✗ Enformer predictions failed: {result.stderr}")

In [ ]:
print("\nStep 1.4: Enhancer Identification...")
cmd = [sys.executable, str(phase1_dir / 'identify_functional_enhancer.py'),
       '--config', str(config_file), '--target-snp', target_snp]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Enhancer identification completed")
    
    # Display Enformer plot if available
    plot_file = output_dir / f"{target_snp}_enformer_track_differences.png"
    if plot_file.exists():
        from IPython.display import Image, display
        print("\nEnformer Track Differences:")
        display(Image(str(plot_file)))
        
else:
    print(f"✗ Enhancer identification failed: {result.stderr}")

## Phase 2: EpiAgent Validation

In [ ]:
# Run Phase 2 scripts
phase2_dir = workflow_dir / 'phase2_epiagent_validation'
dataset_id = config['data']['scatac_dataset_id']

print("Running Phase 2: EpiAgent Validation")
print("="*50)

# Step 2.1: Data Preprocessing
print("Step 2.1: Data Preprocessing...")
cmd = [sys.executable, str(phase2_dir / 'epiagent_preprocessing.py'),
       '--config', str(config_file), '--dataset-id', dataset_id]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Data preprocessing completed")
else:
    print(f"✗ Data preprocessing failed: {result.stderr}")

In [ ]:
print("\nStep 2.2: Cell Type Identification...")
cmd = [sys.executable, str(phase2_dir / 'epiagent_cell_identification.py'),
       '--config', str(config_file), '--dataset-id', dataset_id]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Cell type identification completed")
    
    # Display cell type plot if available
    plot_file = output_dir / f"{dataset_id}_epiagent_cell_types.png"
    if plot_file.exists():
        print("\nCell Type Identification Results:")
        display(Image(str(plot_file)))
        
else:
    print(f"✗ Cell type identification failed: {result.stderr}")

In [ ]:
print("\nStep 2.3: Enhancer Validation...")
cmd = [sys.executable, str(phase2_dir / 'validate_enhancer_specificity.py'),
       '--config', str(config_file), '--target-snp', target_snp, '--dataset-id', dataset_id]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Enhancer validation completed")
    
    # Display validation plots
    validation_plot = output_dir / f"{target_snp}_enhancer_validation.png"
    if validation_plot.exists():
        print("\nEnhancer Validation Results:")
        display(Image(str(validation_plot)))
        
else:
    print(f"✗ Enhancer validation failed: {result.stderr}")

## Phase 3: Integration & Reporting

In [ ]:
# Run Phase 3: Generate report
phase3_dir = workflow_dir / 'phase3_integration_reporting'

print("Running Phase 3: Integration & Reporting")
print("="*50)

cmd = [sys.executable, str(phase3_dir / 'generate_final_report.py'),
       '--config', str(config_file), '--target-snp', target_snp]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Final report generation completed")
    
    # Display report location
    report_file = output_dir / f"{target_snp}_final_report.md"
    if report_file.exists():
        print(f"\nFinal report saved to: {report_file}")
        
        # Display first few lines of report
        with open(report_file, 'r') as f:
            report_preview = f.readlines()[:20]
        
        print("\nReport Preview:")
        print("-" * 50)
        for line in report_preview:
            print(line.rstrip())
        print("...")
        
else:
    print(f"✗ Report generation failed: {result.stderr}")

## Results Summary

In [ ]:
# Load and display workflow results summary
workflow_state_file = output_dir / 'workflow_state.json'

if workflow_state_file.exists():
    with open(workflow_state_file, 'r') as f:
        workflow_state = json.load(f)
    
    print("Workflow Summary:")
    print("=" * 50)
    print(f"Disease: {workflow_state['disease']}")
    print(f"Target SNP: {workflow_state['target_snp']}")
    print(f"Completion time: {workflow_state['completion_time']}")
    print(f"Phases completed: {workflow_state['phases_completed']}/{workflow_state['total_phases']}")
    print("\nPhase Status:")
    
    for phase, completed in workflow_state['workflow_state'].items():
        status = "✅" if completed else "❌"
        print(f"  {status} {phase}")
        
else:
    print("Workflow state file not found")

In [ ]:
# List all output files
print("\nGenerated Output Files:")
print("-" * 30)

output_files = sorted(list(output_dir.glob('*')))
for i, file_path in enumerate(output_files, 1):
    file_size = file_path.stat().st_size
    if file_size > 1024*1024:  # > 1MB
        size_str = f"{file_size/1024/1024:.1f} MB"
    elif file_size > 1024:  # > 1KB
        size_str = f"{file_size/1024:.1f} KB"
    else:
        size_str = f"{file_size} bytes"
    
    print(f"{i:2d}. {file_path.name} ({size_str})")
    
print(f"\nTotal files: {len(output_files)}")

## Analysis Insights

Based on the completed workflow, we can now examine the key findings:

In [ ]:
# Load key results for interpretation
try:
    # Enhancer analysis results
    enhancer_summary_file = output_dir / f"{target_snp}_enhancer_analysis_summary.json"
    if enhancer_summary_file.exists():
        with open(enhancer_summary_file, 'r') as f:
            enhancer_summary = json.load(f)
        
        print("Candidate Enhancer Analysis:")
        print("-" * 40)
        if enhancer_summary['candidate_enhancer']:
            enh = enhancer_summary['candidate_enhancer']
            print(f"Location: chr{enh['chromosome']}:{enh['start']:,}-{enh['end']:,}")
            print(f"Effect size: {enh['max_effect_size']:.3f}")
            print(f"Direction: {enh['effect_direction']}")
            print(f"Track: {enh['track_type']}")
        else:
            print("No significant candidate enhancer identified")
    
    # Validation results
    validation_file = output_dir / f"{target_snp}_enhancer_validation_results.json"
    if validation_file.exists():
        with open(validation_file, 'r') as f:
            validation_results = json.load(f)
        
        print("\nCell-Type Validation:")
        print("-" * 40)
        if 'specificity_analysis' in validation_results:
            spec = validation_results['specificity_analysis']
            most_specific = spec.get('most_specific_celltype')
            if most_specific:
                print(f"Most specific cell type: {most_specific}")
                if spec.get('significant'):
                    print(f"Statistical significance: p = {spec['pvalue']:.3e}")
                    
        if 'disease_relevance' in validation_results:
            relevance = validation_results['disease_relevance']
            status = "✅ Yes" if relevance['is_disease_relevant'] else "⚠️ No"
            print(f"Disease relevant: {status}")
            print(f"Explanation: {relevance['explanation']}")
            
except Exception as e:
    print(f"Could not load detailed results: {e}")

## Next Steps

Based on these computational predictions, consider the following experimental validation approaches:

1. **CRISPRi/CRISPRa experiments** to directly test enhancer function
2. **Reporter assays** to measure enhancer activity differences
3. **Cell-type specific** functional studies in relevant cell types
4. **Chromatin conformation capture** to identify target genes

## Conclusion

This notebook demonstrated the complete GWAS-Enformer-EpiAgent workflow for functional validation of disease-associated genetic variants. The integrative approach provides computational evidence that can guide targeted experimental validation studies.